# 06. 분류 기초
KNN 분류, LogisticRegression, DecisionTree를 비교하면서 분류 모델의 기본 흐름과 평가 지표를 정리합니다.

> 상관관계 확인, 데이터 분리, 스케일링은 이미 학습한 내용이므로 준비 단계에서 한 번에 처리합니다.

In [1]:
from common.libraries import *

# 이번 학습에서 집중할 기능
# - KNeighborsClassifier : 가까운 이웃으로 클래스 예측
# - LogisticRegression : 확률을 이용한 선형 분류
# - DecisionTreeClassifier : 조건 분기로 클래스를 예측
# - confusion_matrix / accuracy / precision / recall / F1 : 분류 성능 평가

In [ ]:
# 기본 준비: 데이터 로드 → X/y → 8:2 분할 → 표준화
wine = pd.read_csv("./data/08_wine.csv")

X = wine[["alcohol", "sugar", "pH"]]
y = wine["class"]

train_input, test_input, train_target, test_target = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_input)
test_scaled = scaler.transform(test_input)

print(train_scaled.shape, test_scaled.shape)

## 상관관계 빠르게 확인
상관관계가 낮다고 해서 해당 특성을 자동으로 제거하지는 않습니다.  
상관관계는 **선형 관계를 확인하는 참고 지표**이고, 실제 모델 성능 비교와 함께 판단합니다.

In [ ]:
corr = wine.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="Oranges", fmt=".2f")
plt.show()

for column in X.columns:
    pearson_corr, pearson_p = pearsonr(X[column], y)
    spearman_corr, spearman_p = spearmanr(X[column], y)
    print(
        f"{column:8s} | "
        f"Pearson={pearson_corr:.4f} (p={pearson_p:.4g}) | "
        f"Spearman={spearman_corr:.4f} (p={spearman_p:.4g})"
    )

## KNN 분류

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_scaled, train_target)

train_acc = knn.score(train_scaled, train_target)
test_acc = knn.score(test_scaled, test_target)

print("훈련 정확도 :", train_acc)
print("테스트 정확도 :", test_acc)
print("훈련-테스트 차이 :", train_acc - test_acc)

## LogisticRegression
`C`가 작을수록 규제가 강하고, 클수록 규제가 약해집니다.  
`max_iter`는 최적화 반복 횟수의 상한입니다.

In [ ]:
lr = LogisticRegression(C=0.13, max_iter=1000)
lr.fit(train_scaled, train_target)

print("훈련 정확도 :", lr.score(train_scaled, train_target))
print("테스트 정확도 :", lr.score(test_scaled, test_target))

## DecisionTree
결정트리는 거리 기반 모델이 아니므로 스케일링이 필수는 아닙니다.  
여기서는 원본 특성을 사용해 트리의 깊이(`max_depth`)에 따른 변화를 확인합니다.

In [ ]:
depth_results = []

for depth in range(2, 21):
    dt = DecisionTreeClassifier(random_state=42, max_depth=depth)
    dt.fit(train_input, train_target)

    train_acc = dt.score(train_input, train_target)
    test_acc = dt.score(test_input, test_target)

    depth_results.append([depth, train_acc, test_acc, train_acc - test_acc])

depth_df = pd.DataFrame(
    depth_results,
    columns=["max_depth", "훈련정확도", "테스트정확도", "훈련-테스트차이"]
)
depth_df.head()

## 분류 성능 평가

In [ ]:
dt = DecisionTreeClassifier(random_state=42, max_depth=12)
dt.fit(train_input, train_target)

train_pred = dt.predict(train_input)
test_pred = dt.predict(test_input)

cm = confusion_matrix(test_target, test_pred, labels=dt.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dt.classes_).plot()
plt.show()

print("정확도 :", accuracy_score(test_target, test_pred))
print("정밀도 :", precision_score(test_target, test_pred))
print("재현율 :", recall_score(test_target, test_pred))
print("F1    :", f1_score(test_target, test_pred))

### 평가 지표 해석
- **정확도**: 전체 예측 중 맞춘 비율
- **정밀도**: Positive라고 예측한 것 중 실제 Positive 비율
- **재현율**: 실제 Positive 중 모델이 찾아낸 비율
- **F1**: 정밀도와 재현율의 조화평균

어떤 지표를 우선할지는 문제의 목적과 오류 비용에 따라 달라집니다. 항상 재현율이나 정확도 하나만으로 모델을 고르지는 않습니다.